In [ ]:
!pip install timm
!pip install transformers
!pip install pyspellchecker
!pip install torch torchvision

In [ ]:
import torch
# import clip
import argparse
import numpy as np

torch.cuda.empty_cache()
import pandas as pd

# Initializations

In [ ]:
class CFG:
    # If there's a GPU available...
    if torch.cuda.is_available():
      # Tell PyTorch to use the GPU.
      device = torch.device("cuda")
      print('There are %d GPU(s) available.' % torch.cuda.device_count())
      print('We will use the GPU:', torch.cuda.get_device_name())
    # If not...
    else:
      print('No GPU available, using the CPU instead.')
      device = torch.device("cpu")


    image_path = "/kaggle/input/dimemex/train/train/"


    # train_file = '/content/drive/MyDrive/Experiments/Hate Detection with CLIP/train_small.csv'
    # train_file = '/content/drive/MyDrive/Experiments/Hate detection with MyModel/train.csv'
#     train_file = '/kaggle/input/exist-memes-dataset/EXIST 2024 Memes Dataset/EXIST 2024 Memes Dataset/training/EXIST2024_training.json'
    train_file = '/kaggle/input/dimemex/train.csv'
#     train_full_file = "/kaggle/input/dimemex/train_data.csv"

    train_labels_file = '/kaggle/input/dimemex/train-labels_subtask_1.csv'
#     train_labels_file = '/kaggle/input/exist-memes-dataset/EXIST2024_training_task4_majority_class_hard.json'


    # val_file = '/content/drive/MyDrive/Experiments/Hate Detection with CLIP/validation_small.csv'
    # val_file = '/content/drive/MyDrive/Experiments/Hate detection with MyModel/validation.csv'
#     val_file = '/kaggle/input/data-cleaned/val_cleaned.csv'
    val_file = '/kaggle/input/dimemex/val.csv'
#     val_file = '/kaggle/input/exist-memes-dataset/validation_small.json'

    test_file = '/kaggle/input/dimemex/test.csv'
#     test_file = '/kaggle/input/dimemex/test_data.csv'



In [ ]:
#Code for original dataset without splitting
column_names1 = ['filename', 'text']
column_names2 = ['hateful', 'inappropriate', 'none']

# Read the CSV file and assign the column names
df_mex = pd.read_csv(CFG.train_full_file, names=column_names1)
df_labels = pd.read_csv(CFG.train_labels_file, names=column_names2)

# Concatenate the two DataFrames along the columns axis (axis=1)
df_train = pd.concat([df_mex, df_labels], axis=1)

df_train

In [ ]:
hateful_count = df_train['hateful'].value_counts()[1]
inappropriate_count = df_train['inappropriate'].value_counts()[1]
none_count = df_train['none'].value_counts()[1]

print(hateful_count)
print(inappropriate_count)
print(none_count)

# Getting Embeddings from BLIP-2

In [ ]:
df_train = pd.read_csv(CFG.train_file)
df_test = pd.read_csv(CFG.test_file)
df_val = pd.read_csv(CFG.val_file)
df_test

In [ ]:
hateful_count = df_test['hateful'].value_counts()[1]
inappropriate_count = df_train['inappropriate'].value_counts()[1]
none_count = df_train['none'].value_counts()[1]

print(hateful_count)
print(inappropriate_count)
print(none_count)

In [ ]:
!pip install deep-translator

In [ ]:
# from transformers import Blip2Processor, Blip2Model
#Model used for submission
# from transformers import AutoProcessor, BlipForImageTextRetrieval

from deep_translator import GoogleTranslator
from PIL import Image
from torch.utils.data import DataLoader, Dataset

#Model after submission
from transformers import AutoProcessor, BlipForConditionalGeneration

processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


# Load the processor and model
# processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
# model = Blip2Model.from_pretrained("Salesforce/blip2-opt-2.7b")

# model = BlipForImageTextRetrieval.from_pretrained("Salesforce/blip-itm-base-coco")
# processor = AutoProcessor.from_pretrained("Salesforce/blip-itm-base-coco")

class MemeDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        filename = row['filename']
        image = Image.open(CFG.image_path + filename + ".jpg")
        text = row['text']
        text_translated = GoogleTranslator(source='auto', target='en').translate(text)
        inputs = processor(images=image, text=text_translated, return_tensors="pt", padding='max_length', truncation=True, 
            max_length=128)
        label = row[['hateful', 'inappropriate', 'none']].values.astype(float)
        labels = torch.tensor(label, dtype=torch.float32)
        encoding = {k:v.squeeze() for k,v in inputs.items()}
        return encoding

# Create the dataset and dataloader
train_dataset = MemeDataset(df_train)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)

val_dataset = MemeDataset(df_val)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=True)

In [ ]:
model = model.to('cuda')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()

for epoch in range(15):
  print("Epoch:", epoch)
  for idx, batch in enumerate(train_dataloader):
    input_ids = batch.pop("input_ids").to(CFG.device)
    pixel_values = batch.pop("pixel_values").to(CFG.device)

    outputs = model(input_ids=input_ids,
                    pixel_values=pixel_values,
                    labels= input_ids)
    
    loss = outputs.loss

    print("Loss:", loss.item())

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

In [ ]:
all_embeddings = []
all_labels = []

for batch in train_dataloader:
    batch = {k: v.to('cuda') for k, v in batch.items()}
    
     # Check and adjust the shape of the input tensor
    for k, v in batch.items():
            # Remove the extra dimension
        batch[k] = v.squeeze(1)
        print(batch[k].shape)
        
    
    # Forward pass to get embeddings
    with torch.no_grad():
        outputs = model(**batch)
        print(outputs)
    
    # Get the combined embeddings
    combined_embeddings = outputs.last_hidden_state
    all_embeddings.append(combined_embeddings.cpu())
#     all_labels.append(labels.cpu())

# Concatenate all embeddings and labels
all_embeddings = torch.cat(all_embeddings)

all_embeddings.size()

# Applying a classifier

In [ ]:
import torch
import torch.nn as nn

class MemeClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MemeClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

# Example data: assuming `all_embeddings` is a tensor of shape [227, 577, 768]
# You need to reduce the embeddings to shape [227, 768] by averaging or another pooling strategy
pooled_embeddings = all_embeddings.mean(dim=1)  # Example of mean pooling

# Initialize the classifier
input_dim = pooled_embeddings.size(1)  # Should be 768 after pooling
hidden_dim = 256
output_dim = 3  # Number of classes: hate-speech, inappropriate content, none
classifier = MemeClassifier(input_dim, hidden_dim, output_dim).to(CFG.device)

# Example forward pass with the pooled embeddings
output = classifier(pooled_embeddings.to(CFG.device))


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Create a DataLoader for the pooled embeddings and labels
dataset = TensorDataset(pooled_embeddings, all_labels)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Use CrossEntropyLoss for multi-class classification
optimizer = optim.Adam(classifier.parameters(), lr=0.001)

# Training loop
num_epochs = 12

for epoch in range(num_epochs):
    classifier.train()
    total_loss = 0
    for batch_embeddings, batch_labels in dataloader:
        batch_embeddings = batch_embeddings.to(CFG.device)
        batch_labels = batch_labels.to(CFG.device)
        
        # Forward pass
        outputs = classifier(batch_embeddings)
        loss = criterion(outputs, batch_labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader)}")

print("Training complete.")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, accuracy_score

pooled_val_embeddings = val_embeddings.mean(dim=1)
pooled_val_embeddings = pooled_val_embeddings.to(CFG.device)
val_labels = val_labels.to(CFG.device)
pooled_test_embeddings = test_embeddings.mean(dim=1)
pooled_test_embeddings = pooled_test_embeddings.to(CFG.device)

classifier.eval()
with torch.no_grad():
    val_outputs = classifier(pooled_test_embeddings)
    val_outputs = (val_outputs > 0.5).float()  # Apply a threshold to get binary predictions

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Convert PyTorch tensors to numpy arrays if necessary
pooled_embeddings_t = all_embeddings.mean(dim=1)
train_embeddings = pooled_embeddings_t.cpu().numpy()
train_labels = all_labels.cpu().numpy()
pooled_embeddings_v = val_embeddings.mean(dim=1)
val_embeddings_n = pooled_embeddings_v.cpu().numpy()
val_labels_n = val_labels.cpu().numpy()
# pooled_embeddings_t = test_embeddings.mean(dim=1)
# test_embeddings_n = pooled_embeddings_t.cpu().numpy()

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier

# Initialize the base classifier
base_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Wrap it with MultiOutputClassifier
multi_target_rf = MultiOutputClassifier(base_rf, n_jobs=-1)

# Train the model
multi_target_rf.fit(train_embeddings, train_labels)

# Predict on the validation set
val_predictions = multi_target_rf.predict(val_embeddings_n)

# Print classification report for each label
print(classification_report(val_labels_n, val_predictions, target_names=['hateful', 'inappropriate', 'none']))

# Calculate accuracy for each label
label_accuracies = (val_predictions == val_labels_n).mean(axis=0)
print(f"Accuracy for each label: {label_accuracies}")

# Overall accuracy (proportion of correctly predicted label sets)
overall_accuracy = (val_predictions == val_labels_n).all(axis=1).mean()
print(f"Overall accuracy: {overall_accuracy * 100:.2f}%")

In [ ]:
test_predictions = multi_target_rf.predict(test_embeddings_n)

val_outputs = val_outputs.cpu().numpy()
test_predictions = val_outputs

In [ ]:
# Convert predictions from float to integer
test_predictions_int = test_predictions.astype(int)

# Convert to a DataFrame
predictions_df = pd.DataFrame(test_predictions_int)

# Save to CSV file
predictions_df.to_csv('/kaggle/working/predictions2.csv', index=False, header=False)

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score


In [ ]:
# Initialize and train the model
gb = GradientBoostingClassifier(n_estimators=100)
gb.fit(train_embeddings, train_labels)

# Predict and evaluate
val_predictions = gb.predict(val_embeddings_n)
print(classification_report(val_labels_n, val_predictions))
print("Accuracy:", accuracy_score(val_labels_n, val_predictions))

column_names1 = ['filename', 'text']
df_test = pd.read_csv(CFG.test_file, names=column_names1)

test_dataset = MemeDataset(df_test)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
test_embeddings = []
# test_labels = []

for batch in test_dataloader:
    batch = {k: v.to('cuda') for k, v in batch.items()}
    
     # Check and adjust the shape of the input tensor
    for k, v in batch.items():
        batch[k] = v.squeeze(1)
    
    # Forward pass to get embeddings
    with torch.no_grad():
        outputs = model(**batch)
    
    # Get the combined embeddings
    combined_embeddings = outputs.last_hidden_state
    test_embeddings.append(combined_embeddings.cpu())

# Concatenate all embeddings and labels
test_embeddings = torch.cat(test_embeddings)

In [ ]:
pooled_embeddings_t = test_embeddings.mean(dim=1)
test_embeddings_n = pooled_embeddings_t.cpu().numpy()

# Initialize and train the model
gb = GradientBoostingClassifier(n_estimators=100)
gb.fit(train_embeddings, train_labels)

# Predict and evaluate
test_predictions = gb.predict(test_embeddings_n)